# BLUE - MSL
Optimal interpolation of along-track SLA data to create SLA grids. Along-track 1 Hz uncertainties are propagated to MSL grids through the inverse method.

The method relies on the BLUE theory (Best Linear Unbiased Estimator) to combine SLA along-track data and the associated uncertainties to regional mean sea level grids with the corresponding uncertainties. The output grids can have arbitrary spatial and temporal resolutions.

## Hypotheses to discuss

- The shape of the correlation kernel for the background B (gaussian, with typical spatial and temporal scales, constant)
- The values of the typical scales of the background (time and space)
- The shape of the observation errors covariance matrix, with a filling method similar to an error budget
- The content of this error budget, supposed to be identical for all passes and all portions of each pass
- The spatio-temporal selection of data within margins of 3 times the typical scales
- The sub-sampling of 1 point over 3 (or 5) to speed up the calculation

## Imports

In [ ]:
# Imports
import numpy as np, xarray as xr, matplotlib.pyplot as plt, pandas as pd
import os
os.environ['ESMFMKFILE'] = "/work/ODATIS/GRACEFUL/F3O/virtual_env/f3o_env_2/lib/esmf.mk"
import netCDF4, lenapy, glob, time
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.linalg import solve, cho_factor, cho_solve
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import dask
import dask.array as da
from dask import delayed
from dask.distributed import Client, LocalCluster

## Functions

### Covariance matrices modelisation

In [ ]:
def correlation_kernel(coords1, coords2, L, tau, sigma_b):
    """
    Computes the spatio-temporal covariance matrix of the background B (or portions of B).
    This function defines the statistical structure of the oceanic signal (typically the background).
    
    Arguments:
        coords1 (np.ndarray): Matrix of shape (N, 3) containing [lon, lat, time] of the first dataset.
        coords2 (np.ndarray): Matrix of shape (M, 3) containing [lon, lat, time] of the second dataset.
        L (float): Spatial decorrelation scale (in degrees).
        tau (float): Temporal decorrelation scale (in days).
        sigma_b (float): Standard deviation of the oceanic signal (expected natural variability, in m).
        
    Returns:
        np.ndarray: Covariance matrix of shape (N, M).
    """
    # If one of the datasets is empty, returns an empty matrix to avoid errors from cdist
    if coords1.shape[0] == 0 or coords2.shape[0] == 0:
        return np.array([[]])
    
    # Calculation of the pairwise distances matrices
    # d_space is composed of the euclidian distances (in degrees) between all the points
    d_space = cdist(coords1[:, :2], coords2[:, :2])
    # d_time is composed of the temporal distances (in days) between all the points
    d_time = cdist(coords1[:, 2:], coords2[:, 2:])
    
    # Spatio-temporal covariance model (Gaussian Kernel)
    # The longer the distance (spatial or temporal), the closer the correlation is to 0.
    B = (sigma_b**2) * np.exp(-(d_space**2) / (2 * L**2)) * np.exp(-(d_time**2) / (2 * tau**2))
    
    return B


In [ ]:
def build_complex_R(time_obs, error_specs):
    """
    Builds the observation errors covariance matrix R.
    It can combine several types of errors, listed in an uncertainty budget
    (white instrumental noise, short/long-term time-correlated errors)
    
    Arguments:
        time_obs (np.ndarray): Vector of shape (N,) containing the observation times (in days).
        error_specs (list of dict): List error components.
            Ex: [{'name': 'white', 'sigma': 0.02, 'tau': 0}, {'name': 'long', 'sigma': 0.01, 'tau': 180}]
            
    Returns:
        np.ndarray: Error covariance matrix of shape (N, N).
    """
    n_obs = len(time_obs)
    # Initialisation of the R matrix with zeros
    R = np.zeros((n_obs, n_obs))
    
    # Reshape the time vector as a column (N, 1) to be compatible with cdist
    t_coords = time_obs.reshape(-1, 1)
    # Calculation of the absolute time differences matrix between each observation
    dist_t = cdist(t_coords, t_coords)

    # Iteration on every error component specified by the user
    for spec in error_specs:
        sigma = spec['sigma']
        tau_err = spec['tau']
        
        if tau_err == 0:
            # If tau = 0, the error is purely random and independent between each measurement (white noise).
            # So we add the variance (sigma^2) only on the diagonal.
            R += np.eye(n_obs) * (sigma**2)
        else:
            # If tau > 0, the error is corrated in time.
            # So we use an exponential decay model.
            R += (sigma**2) * np.exp(-dist_t / tau_err)
            
    return R


### Processing of a tile in the grid

In [ ]:
def process_grid_chunk_optimized(lon_chunk, lat_chunk, obs_lon, obs_lat, obs_time, obs_ssh, L, tau, sigma_b, error_config):
    """
    Computes a SLA grid chunk. This function is meant to be run on a single dask worker.
    
    Arguments:
        lon_chunk (np.ndarray): Vector of shape (Ngrid_lon,) containing the longitude values of the grid chunk to compute.
        lat_chunk (np.ndarray): Vector of shape (Ngrid_lat,) containing the latitude values of the grid chunk to compute.
        obs_lon (np.ndarray): Vector of shape (Nobs,) containing the longitude values of the observation points to process.
        obs_lat (np.ndarray): Vector of shape (Nobs,) containing the latitude values of the observation points to process.
        obs_time (np.ndarray): Vector of shape (Nobs,) containing the time values of the observation points to process.
        obs_ssh (np.ndarray): Vector of shape (Nobs,) containing the SLA values of the observation points to process.
        L (float): Typical spatial correlation scale in km (to be used both for the correlation kernel and the space selection margin).
        tau (float): Typical temporal correlation scale in days (to be used both for the correlation kernel and the time selection margin).
        sigma_b (float): Standard deviation of the background in m (to be used for the correlation kernel).
        error_config (list of dict): List of error components for the observation error covariance matrix R.
            Ex: [{'name': 'white', 'sigma': 0.02, 'tau': 0}, {'name': 'long', 'sigma': 0.01, 'tau': 180}]
            
    Returns:
        np.ndarray: The analysed SLA grid chunk and the corresponding uncertainty.
    """
    # Build empty SLA and SLA_unc grids
    ssh_chunk = np.full((len(lat_chunk), len(lon_chunk)), np.nan)
    err_chunk = np.full((len(lat_chunk), len(lon_chunk)), np.nan)

    # Define the selection margins as 3 times the typical scales to select only useful data for the analysis.
    # Data outside the margins won't be taken into account
    margin_space = 3 * L
    margin_time = 3 * tau

    # Select only relevant data for the tile
    min_lon, max_lon = lon_chunk.min(), lon_chunk.max()
    min_lat, max_lat = lat_chunk.min(), lat_chunk.max()
    
    center_lon = (min_lon + max_lon) / 2 # Specific case of longitudes (need to handle the 0°-360° transition)
    dist_lon_chunk = np.minimum(np.abs(obs_lon - center_lon), 360 - np.abs(obs_lon - center_lon))
    
    mask_chunk = (
        (obs_lat >= min_lat - margin_space) & (obs_lat <= max_lat + margin_space) & # Select latitude data according to margins
        (dist_lon_chunk <= ((max_lon - min_lon) / 2 + margin_space)) & # Select longitude data according to margins
        (np.abs(obs_time) <= margin_time) # Select time data according to margins
    )

    # Keep only one point out of three to fasten the processing
    step = 3
    c_lon = obs_lon[mask_chunk][::step]
    c_lat = obs_lat[mask_chunk][::step]
    c_time = obs_time[mask_chunk][::step]
    c_ssh = obs_ssh[mask_chunk][::step]
    
    # Filter abnormal SLA values
    valid = np.isfinite(c_ssh) & (np.abs(c_ssh) < 5.0)
    c_lon, c_lat, c_time, c_ssh = c_lon[valid], c_lat[valid], c_time[valid], c_ssh[valid]

    # Return NaN if there is less than 3 remaining points for the analysis (should not occur often)
    if len(c_ssh) < 3:
        return ssh_chunk, err_chunk

    # Inversion for the tile
    coords_obs = np.stack([c_lon, c_lat, c_time], axis=1)
    
    R = build_complex_R(c_time, error_config) # Observation errors covariance matrix
    B_obs = correlation_kernel(coords_obs, coords_obs, L, tau, sigma_b) # Background error covariance matrix in the observation space (HBH^T)
    Cy = B_obs + R
    
    try:
        # Cholesky factorisation: fast for positive matrices
        c, lower = cho_factor(Cy)
        # Compute the weights w (Cy * w = y); w = [(HBH^T+R)^-1]y; xa = (BH^T)w
        w = cho_solve((c, lower), c_ssh)
    except Exception: # If an error occurs in the inversion, return NaN
        return ssh_chunk, err_chunk

    # Projection on the grid
    for i, lat in enumerate(lat_chunk):
        for j, lon in enumerate(lon_chunk):
            
            target_coord = np.array([[lon, lat, 0.0]]) # Time is handled as a difference with target date
            
            # Correlation vector between grid and observations (= BH^T)
            C_grid_obs = correlation_kernel(target_coord, coords_obs, L, tau, sigma_b)
            
            # Estimated SLA; xa = (BH^T)w
            ssh_chunk[i, j] = (C_grid_obs @ w).item()
            
            # Estimated variance (Cy^-1 * C_grid_obs.T via Cholesky)
            C_inv_Cg = cho_solve((c, lower), C_grid_obs.T)
            # A = B - KHB = B - C_grid_obs.Cy^-1.C_grid_obs^T
            err_var = sigma_b**2 - np.sum(C_grid_obs * C_inv_Cg.T)
            
            err_chunk[i, j] = np.sqrt(max(0, err_var)) # Corresponding uncertainty, set to 0 if negative variance

    return ssh_chunk, err_chunk

### Construction of the dask graph

In [ ]:
def beg_end_dates_from_l2p_filename(f):
    """
    Returns the beginning and end dates of a pass from the L2P filename.
    
    Arguments:
        f (str): The complete, official and formatted altimetry track filename.
            
    Returns:
        tuple of np.datetime64: The beginning and end dates of the altimetry pass.
    """
    beg_date_str = f.split('_P')[1][5:20]
    end_date_str = f.split('_P')[1][21:36]
    beg_date_dt = np.datetime64(f"{beg_date_str[:4]}-{beg_date_str[4:6]}-{beg_date_str[6:11]}:{beg_date_str[11:13]}:{beg_date_str[13:15]}")
    end_date_dt = np.datetime64(f"{end_date_str[:4]}-{end_date_str[4:6]}-{end_date_str[6:11]}:{end_date_str[11:13]}:{end_date_str[13:15]}")
    return beg_date_dt, end_date_dt

In [ ]:
def run_dask_oi_process(obs_ds, grid_lon, grid_lat, target_time, L, tau, sigma_b, error_config, margin_time, chunk_size=10):
    """
    Main function that orchestrates the distributed computation. It splits the output grid into tiles
    and builds the dask task graph.
    
    Arguments:
        obs_ds (xr.Dataset): The dataset with all selected observations.
        grid_lon (np.ndarray): The list of longitude values on the grid.
        grid_lat (np.ndarray): The list of latitude values on the grid.
        target_time (np.datetime64): The target center date of the grid.
        L (float): Typical spatial correlation scale in km (to be used both for the correlation kernel and the space selection margin).
        tau (float): Typical temporal correlation scale in days (to be used both for the correlation kernel and the time selection margin).
        sigma_b (float): Standard deviation of the background in m (to be used for the correlation kernel).
        error_config (list of dict): List of error components for the observation error covariance matrix R.
            Ex: [{'name': 'white', 'sigma': 0.02, 'tau': 0}, {'name': 'long', 'sigma': 0.01, 'tau': 180}]
        margin_time (float): Temporal margin before and after target time.
            Files outside the target_time +- margin_time will not be used.
        chunk_size (int): Size of the tiles: chunk_size=10 (default value) means that each task computes 10x10 points block.
            The chunk size is the same for latitude and longitude dimensions.
            
    Returns:
        tuple of dask delayed xr.DataArray: The final dask objects with all planned operations.
            Using dask.compute() on these objects will launch the actual execution on the cluster.
    """

    # First, we load into memory the observation dataset so that several workers don't need to read the same file at the same time.
    print("Loading and preparation of the observations into memory...")
    t_start = target_time - np.timedelta64(int(margin_time), 'D')
    t_end = target_time + np.timedelta64(int(margin_time), 'D')
    ds_sub = obs_ds.sel(time=slice(t_start, t_end)).compute() # Temporal selection and loading
    
    # Clear all the NaNs in the SLA variable
    ds_sub = ds_sub.dropna(dim='time', subset=['sea_level_anomaly'])
    
    # Normalisation of the dates into floats to allow the cdist function to work on time values
    time_norm = (ds_sub.time - np.datetime64(target_time)) / np.timedelta64(1, 'D')
    
    # Extract the values in numpy arrays (no need for each worker to perform this operation)
    obs_lon_np = ds_sub.longitude.values
    obs_lat_np = ds_sub.latitude.values
    obs_time_np = time_norm.values
    obs_ssh_np = ds_sub.sea_level_anomaly.values

    # Second, we chunk the grid into tiles
    print("Building up the Dask graph...")
    # Split the coordinates vectors into smaller segmets
    lon_chunks = np.array_split(grid_lon, max(1, len(grid_lon) // chunk_size))
    lat_chunks = np.array_split(grid_lat, max(1, len(grid_lat) // chunk_size))

    # Third, we create delayed tasks and convert them into dask arrays
    ssh_blocks = []
    err_blocks = []
    for lat_chunk in lat_chunks: # Loop through all tiles
        ssh_row = []
        err_row = []
        for lon_chunk in lon_chunks:
            # Creation of the delayed inversion task
            res = delayed(process_grid_chunk_optimized, nout=2)(
                lon_chunk, lat_chunk, 
                obs_lon_np, obs_lat_np, obs_time_np, obs_ssh_np, 
                L, tau, sigma_b, error_config
            )

            # Conversion of the delayed object into a dask array
            chunk_shape = (len(lat_chunk), len(lon_chunk))
            ssh_da_chunk = da.from_delayed(res[0], shape=chunk_shape, dtype=np.float64)
            err_da_chunk = da.from_delayed(res[1], shape=chunk_shape, dtype=np.float64)
            
            # Store the tiles for each latitude row
            ssh_row.append(ssh_da_chunk)
            err_row.append(err_da_chunk)
            
        # Store each row to create a 2D grid
        ssh_blocks.append(ssh_row)
        err_blocks.append(err_row)

    # Fourth and finally, we gather the results
    ssh_da = da.block(ssh_blocks)
    err_da = da.block(err_blocks)

    return ssh_da, err_da # Return the delayed dask arrays

## Computation

### Open dask cluster

In [ ]:
# Local cluster if the JupyterHub session is large enough
cluster = LocalCluster(n_workers=32)
client = Client(cluster)
client

In [ ]:
# Slurm cluster otherwise
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(                      # Dask-worker specific keywords
    n_workers=32,                   # number of workers
    cores=1,                        # each worker runs on <nb_cores> cores
    memory="16GB",                 # each worker uses <N*8>GB memory (on TREX g2022 : nb_cores*8Go, on g2019 : nb_cores*4.6Go )
    processes=1,                   # Number of Python processes to cut up each job
    local_directory='$TMPDIR',               # Location to put temporary data if necessary
    account='cnes_level2',              # Project account
    walltime='02:00:00',                   # Computation time for worker
    interface='ib0',                         # InfiniBand
    log_directory='../dask-logs',            # Dask logs
    job_extra_directives=['--qos="cpu_2022_64"'],  # qos to use
)
client = Client(cluster)
client

### Launch script

In [ ]:
# Background parametres: will be used for the correlation kernel and the spatio-temporal margins
L = 3.0 # in degrees
tau = 10.0 # in days
sigma_b = 0.1 # in m

# Select relevant time window by only selecting required files
target_time = np.datetime64('2025-09-22') # TO MODIFY
margin_time = np.timedelta64(5, 'D') # TO MODIFY # Represents half the temporal resolution of the final grid

path_data = '/work/ODATIS/MOHeaCAN/data/altimetry/l2p_data/s6a/data_store/' # TO MODIFY # Location of the L2P files

# Restrict the list of files to the targeted temporal resolution
beg_date, end_date = target_time - margin_time, target_time + margin_time
list_files = sorted(glob.glob(path_data + 'global_sla_l2p_ntc_s6a_lr_C*_P*_*T*_*T*_*T*version_04_00.nc'))
restricted_list_files = []
for f in list_files:
    beg_date_dt, end_date_dt = beg_end_dates_from_l2p_filename(f)
    if beg_date_dt >= beg_date and end_date_dt <= end_date:
        restricted_list_files.append(f)
restricted_list_files = sorted(restricted_list_files)
print(f'Relevant files found: {len(restricted_list_files)}')

# We only load the necessary variables
obs_ds = xr.open_mfdataset(restricted_list_files, engine='h5netcdf')[['longitude', 'latitude', 'sea_level_anomaly', 'validation_flag']]
obs_ds = obs_ds.where(obs_ds['validation_flag'] == 0)
print('Dataset read and ready to be processed')

# Targeted spatial resolution
res_lon, res_lat = 3.0, 1.0 # in degrees
grid_lon = np.arange(0, 360, res_lon)
grid_lat = np.arange(-90, 90, res_lat)

# Parametrisation of the observation errors to build the R covariance matrix
# Examples based on the satellite ground speed are provided but anything is possible
#ground_speed = np.sqrt(398600/(1336 + 6371))
#correlation_spatial_scale = 200
#correlation_time_scale = (correlation_spatial_scale / ground_speed) / (24*3600)
error_config = [
    {'name': 'white_noise', 'sigma': 0.0001, 'tau': 0}, # The white noise (tau=0) can be useful for the numerical stability of the inversion
    #{'name': 'short_term', 'sigma': 0.01, 'tau': correlation_time_scale}, # Example of along-track time-correlated error
]

# Launch the contruction of the dask graph
# chunk_size=5 means that each task computes 5x5 points block
# As the list of files has been restricted above, the margin_time argument is useful only if it is smaller than the above-defined margin_time
ssh_da, unc_da = run_dask_oi_process(
    obs_ds, grid_lon, grid_lat, target_time, L, tau, sigma_b, error_config, margin_time=3*tau, chunk_size=5
)

In [ ]:
# Check what the dask object looks like
ssh_da

In [ ]:
# Launch the computation on the cluster
start = time.time()
print("Start distributed computation...")
ssh_grid, unc_grid = dask.compute(ssh_da, unc_da)
print("End distributed computation...")
end = time.time()
# Print the computation time
print(f'Computation time: {end-start} s = {(end-start)/60} min = {(end-start)/3600} h')

In [ ]:
# Wrap the results in a xr.Dataset object
output = xr.Dataset(
    {
        "ssha": (("lat", "lon"), ssh_grid),
        "ssha_unc": (("lat", "lon"), unc_grid),
    },
    coords={"lat": grid_lat, "lon": grid_lon, "time": target_time}
)

### Display the results

In [ ]:
# Open a Geomask to extract the ocean mask
ds_mask = xr.open_dataset('/home/uz/vaujout/dev/climate_study_tools/data/GEO_mask_1deg_20210406.nc', engine='h5netcdf')
da_mask = ds_mask['EarthMask']
da_mask = da_mask.interp(coords={'latitude':output.lat, 'longitude':output.lon})
# Plot the gridded SSH
plt.figure()
output['ssha'].where(da_mask == 0.).plot(vmin=-0.5, vmax=0.5)
# Plot the corresponding SSH uncertainty
plt.figure()
output['ssha_unc'].where(da_mask == 0.).plot(vmin=0, vmax=0.01)

### Close the cluster

In [ ]:
cluster.close()
client.close()